# Atmospheric River Data Access

Loading and preparing ERA5 Reanalysis Data for visualization via [**PIKART**](https://ar.pik-potsdam.de/?a=general).

---

## Overview
This notebook focuses on preparing data from ERA5 Reanalysis, curated by {cite:t}`pikart:2025` through detection algorithms to focus on Atmospheric Rivers. The datasets provided have global coverage from 1940 to 2023 with 0.5° resolution every 6 hours. We will examine the datasets features and dimensions for accessing specific data within the dataset.

1. Data access
2. Data loading
3. Data examination

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [Xarray](https://docs.xarray.dev/en/stable/) | Necessary | Used to load cloud-stored datasets|
| [Understanding of NetCDF](https://foundations.projectpythia.org/core/data-formats/netcdf-cf) | Helpful | Familiarity with metadata structure |

- **Time to learn**: 20 min

---

## Imports

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

## Data Access

Data can be accessed yearly through a url to a cloud-hosted NetCDF file.

In [2]:
# Year to be loaded (1950-2023)
year = '1996'

# Construct data access url (via THREDDS server)
url = f"https://ar.pik-potsdam.de/thredds/dodsC/eulerian_era5/PIKARTV1_eulerian_ERA5_0p5deg_6hr_{year}.nc"

## Data Loading

In [3]:
%%time
# Open the dataset with xarray
ds = xr.open_dataset(url)
print(f'size: {ds.nbytes / (1024 ** 3)} GB')

size: 7.068172127008438 GB
CPU times: user 10.2 s, sys: 418 ms, total: 10.6 s
Wall time: 24 s


## Data Examination

Below we examine the dataset loaded. Note that it has three dimensions/coordinates: time, latitude, and longitude. These coordinates are used to subquery the dataset for more specific data access.

Time has 1464 values, every 6 hours for the year of 1996 (4 times a day * 366 days = 1464), note that 1996 is a leap-year! Latitude has 360 values at 0.5 ° resolution (90 °S to 90 °N = 180 * 2 = 360) and latitude has 720 (360 ° * 2).

Underneath the coordinates we find the data variables
1. ar_mask
1. duration
1. intensity
1. ivt
1. ivtu
1. ivtv
1. rank

We will examine these below.

Lastly comes the attributes which gives the source of the dataset.

In [11]:
# Examine the dataset
ds

<xarray.Dataset> Size: 8GB
Dimensions:    (time: 1464, latitude: 360, longitude: 720)
Coordinates:
  * time       (time) datetime64[ns] 12kB 1996-01-01T03:00:00 ... 1996-12-31T...
  * latitude   (latitude) float32 1kB -89.75 -89.25 -88.75 ... 88.75 89.25 89.75
  * longitude  (longitude) float32 3kB -180.0 -179.5 -179.0 ... 179.0 179.5
Data variables:
    ar_mask    (time, latitude, longitude) uint8 379MB ...
    duration   (time, latitude, longitude) int16 759MB ...
    intensity  (time, latitude, longitude) float32 2GB ...
    ivt        (time, latitude, longitude) float32 2GB ...
    ivtu       (time, latitude, longitude) float32 2GB ...
    ivtv       (time, latitude, longitude) float32 2GB ...
    rank       (time, latitude, longitude) int8 379MB ...
Attributes:
    conventions:  CF-1.12
    title:        PIK Atmospheric River Trajectories (PIKART) Eulerian Catalo...
    version:      1.0
    source:       Atmospheric river detection tool applied to ERA5 reanalysis...
    institution:  Potsdam Institute for Climate Impact Research (PIK) and Lei...
    references:   Vallejo-Bernal, S. M., Braun, T., Marwan, N., & Kurths, J. ...
    comment:      This dataset contains physical properties during atmospheri...
    history:      Sun Jun 29 18:41:14 2025: ncks -4 -L 1 /p/projects/climxtre...
    NCO:          netCDF Operators version 5.3.0 (Homepage = http://nco.sf.ne...

### ar_mask
Below we can see that the ar_mask variable represents a mask of where an Atmospheric River is present at the globe. For each resolution marker, either a 1 is present if there is an Atmospheric River at that grid point, or a 0 if not.

In [19]:
print(ds.ar_mask.attrs["long_name"])

Binary indicator equal to 1 for atmospheric river conditions


### duration
This shows how long each Atmospheric River persists in units of hours.

In [22]:
print(ds.duration.attrs["long_name"])

Duration of uninterrupted atmospheric river conditions


### intensity
Intensity is the momentary strength of the AR measured by the maximum value of IVT (see below) present within the atmospheric river.

In [4]:
print(ds.intensity.attrs["long_name"])

Maximum vertically integrated water vapor transport during uninterrupted atmospheric river conditions


### ivt
IVT stands for Integrated Vertical Transport of water vapor, commonly used to identify atmospheric rivers. This represents the amount of water vapor moving through a vertical column of air by combining the amount of water vapor in the air column with the wind speeds across the column.

In [25]:
print(ds.ivt.attrs["long_name"])

Magnitude of the vertically integrated water vapor transport during atmospheric river conditions


### ivtu and ivtv
These represent the components of IVT, described above. Separated into *u*, the eastward component and *v*, the northward component.

In [26]:
print("ivtu: " + ds.ivtu.attrs["long_name"])
print("ivtv: " + ds.ivtv.attrs["long_name"])

ivtu: Eastward component of the vertically integrated water vapor transport during atmospheric river conditions
ivtv: Northward component of the vertically integrated water vapor transport during atmospheric river conditions


### rank
Rank is a score 1-5 rating the strength of each atmospheric river, based on the widely implemented AR Scale developed by {cite:t}`Ralph:2019`. Computed by the amount of water vapor transport (IVT) and duration of the AR.

In [30]:
print(ds["rank"].attrs["long_name"])

Rank of uninterrupted atmospheric river conditions according to the strength scale by Ralph et al., (2019)


---

## Summary
This notebook provided a high level overview of accessing datasets through the PIKART catalogue via a PIKART THREDDS server url. Covers a summary the coordinates, dimensions, and variables present within a dataset and their use in Atmospheric River detection. This is a foundational notebook for PIKART data analysis, which will be built on in future notebooks.

### What's next?
This notebook provides a basis for the next notebook: visualization.ipynb. The next notebook uses the basics taught in this notebook to create interesting visualizations of Atmospheric Rivers from datasets accessed through PIKART.

## References
Ralph, F. M., J. J. Rutz, J. M. Cordeira, M. Dettinger, M. Anderson, D. Reynolds, L. J. Schick, and C. Smallcomb, 2019: A Scale to Characterize the Strength and Impacts of Atmospheric Rivers. Bull. Amer. Meteor. Soc., 100, 269–289, https://doi.org/10.1175/BAMS-D-18-0023.1. 

S. M. Vallejo-Bernal, T. Braun, N. Marwan, J. Kurths: PIKART: A Comprehensive Global Catalog of Atmospheric Rivers, Journal of Geophysical Research: Atmospheres, in press, DOI: 10.1029/2024JD041869. 